In [1]:
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained(
    "huggingface-course/bert-base-uncased-tokenizer-without-normalizer",
)

tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [4]:
text = 'here is a sentence adapted to our tokenizer'
print(f'tokenizer.tokenize(text): {tokenizer.tokenize(text)}')

tokenizer.tokenize(text): ['here', 'is', 'a', 'sentence', 'adapted', 'to', 'our', 'token', '##izer']


In [5]:
text = '这是一个不适配到我们的tokenizer的句子'
print(f'tokenizer.tokenize(text): {tokenizer.tokenize(text)}')

tokenizer.tokenize(text): ['[UNK]', '[UNK]', '一', '[UNK]', '不', '[UNK]', '[UNK]', '[UNK]', '我', '[UNK]', '的', 'token', '##izer', '的', '[UNK]', '子']


In [6]:
text = 'the medical vocabulary is divided into many sub-token: paracetamol, phrayngitis'
print(f'tokenizer.tokenize(text): {tokenizer.tokenize(text)}')

tokenizer.tokenize(text): ['the', 'medical', 'vocabulary', 'is', 'divided', 'into', 'many', 'sub', '-', 'token', ':', 'para', '##ce', '##tam', '##ol', ',', 'ph', '##ray', '##ng', '##itis']


# 训练tokenizer的步骤

1. 准备一个用于训练的语料库
2. 选择tokenizer架构
3. 使用语料库训练tokenizer
4. 保存结果

In [7]:
from datasets import load_dataset

raw_datasets = load_dataset('code_search_net', 'python')

README.md: 0.00B [00:00, ?B/s]

python/train-00000-of-00001.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

python/test-00000-of-00001.parquet:   0%|          | 0.00/28.7M [00:00<?, ?B/s]

python/validation-00000-of-00001.parquet:   0%|          | 0.00/30.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/412178 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22176 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/23107 [00:00<?, ? examples/s]

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 412178
    })
    test: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 22176
    })
    validation: Dataset({
        features: ['repository_name', 'func_path_in_repository', 'func_name', 'whole_func_string', 'language', 'func_code_string', 'func_code_tokens', 'func_documentation_string', 'func_documentation_tokens', 'split_name', 'func_code_url'],
        num_rows: 23107
    })
})

In [9]:

def get_training_corpus():
    for start_idx in range(0, len(raw_datasets['train']), 1000):
        samples = raw_datasets["train"][start_idx : start_idx + 1000]
        yield samples["whole_func_string"]

In [10]:
from transformers import AutoTokenizer

train_corpus = get_training_corpus()

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

new_tokenizer = old_tokenizer.train_new_from_iterator(train_corpus, 52000)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [20]:
val_str = raw_datasets['test'][5]['whole_func_string']
print(val_str)

def dictify(r,root=True):
    """http://stackoverflow.com/a/30923963/2946714"""
    if root:
        return {r.tag : dictify(r, False)}
    d=copy(r.attrib)
    if r.text:
        d["_text"]=r.text
    for x in r.findall("./*"):
        if x.tag not in d:
            d[x.tag]=[]
        d[x.tag].append(dictify(x,False))
    return d


In [21]:
print(f'old_tokenizer({val_str}) = {old_tokenizer.tokenize(val_str)}')
print(f'new_tokenizer({val_str}) = {new_tokenizer.tokenize(val_str)}')

old_tokenizer(def dictify(r,root=True):
    """http://stackoverflow.com/a/30923963/2946714"""
    if root:
        return {r.tag : dictify(r, False)}
    d=copy(r.attrib)
    if r.text:
        d["_text"]=r.text
    for x in r.findall("./*"):
        if x.tag not in d:
            d[x.tag]=[]
        d[x.tag].append(dictify(x,False))
    return d) = ['def', 'Ġdict', 'ify', '(', 'r', ',', 'root', '=', 'True', '):', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġ"""', 'http', '://', 'stack', 'over', 'flow', '.', 'com', '/', 'a', '/', '309', '23', '96', '3', '/', '29', '467', '14', '"""', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġif', 'Ġroot', ':', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġreturn', 'Ġ{', 'r', '.', 'tag', 'Ġ:', 'Ġdict', 'ify', '(', 'r', ',', 'ĠFalse', ')}', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġd', '=', 'copy', '(', 'r', '.', 'att', 'rib', ')', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġif', 'Ġr', '.', 'text', ':', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġd', '["', '_', 'text', '"]', '=', 'r', '.', 'text', 'Ċ', 'Ġ', 'Ġ', 'Ġ', 'Ġfor', 'Ġx', 'Ġin', 'Ġr',